In [18]:
from ireal_transformer import load_ireal_transformer
from parser import Tune
import torch.nn.functional as F

In [2]:
data = open("data/data.txt", encoding="utf-8").read().strip()

tunes = Tune.parse_ireal_url(data, verbose=False)

In [3]:
model = load_ireal_transformer("ireal_transformer_model.pth")

/home/ottenl/projects/jazz-transformer/ireal_transformer.py:100: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=layers)


In [6]:
embeddings = [model.get_song_embedding(tune, causal=False) for tune in tunes]

In [ ]:
print(f"Parsed {len(tunes)} tunes.")
print(f"First tune: {tunes[0].title}")
print(f"Embedding shape: {embeddings[0].shape}")
print(embeddings[0])

Parsed 1955 tunes.
First tune: 9.20 Special
Embedding shape: torch.Size([128])
tensor([-1.2449e+00,  1.2527e+00,  1.5732e-01,  2.6571e+00, -2.3590e+00,
         9.6173e-01,  4.1191e-01, -8.8740e-02,  2.9953e-01, -6.8846e-01,
         3.3871e-01, -1.4079e+00, -1.1977e+00,  1.3352e+00, -1.1488e+00,
        -5.3462e-01, -2.3563e+00,  3.1525e-01,  2.9905e-04, -2.1925e-01,
        -2.4713e+00, -3.5051e+00, -1.1520e+00, -1.9960e+00,  4.8781e-01,
         1.0580e+00, -4.8988e-01,  1.5459e+00, -1.0549e+00,  1.3681e+00,
        -1.6064e+00, -2.6800e+00, -1.6880e+00, -6.3456e-01,  5.2941e-01,
        -1.4242e+00, -1.1791e+00, -1.2052e+00,  3.0195e-01,  6.2330e-01,
        -1.2075e+00,  2.1132e-01,  7.8869e-01, -6.4127e-01, -1.5500e+00,
         7.8109e-02, -9.2503e-01,  6.5201e-01, -3.1092e-01, -7.6426e-01,
        -1.0815e+00,  7.2431e-01,  1.6632e+00, -3.0957e-01, -6.8663e-01,
        -4.7677e-01,  3.4851e-01, -4.4369e-01,  1.5384e+00, -2.0224e+00,
        -4.5186e-01, -8.7269e-01, -6.2002e-02

In [10]:
import pandas as pd
df = pd.DataFrame()
for i, tune in enumerate(tunes):
    df = pd.concat([df, pd.DataFrame({
        "title": [tune.title],
        "embedding": [embeddings[i].numpy()]
    })], ignore_index=True)

title                                        How High The Moon
embedding    [-1.36297, -0.744271, 1.501025, 2.700562, -0.5...
Name: 858, dtype: object

In [25]:
song = df[df["title"] == "Giant Steps"].iloc[0]

song_embedding = torch.as_tensor(song["embedding"], dtype=torch.float32)
embeddings_tensor = torch.stack(embeddings)

df[f"similarity_to_{song['title']}"] = F.cosine_similarity(
    embeddings_tensor,
    song_embedding.unsqueeze(0),
    dim=1
).cpu().numpy()

df.sort_values(
    by=f"similarity_to_{song['title']}",
    ascending=False
).head(20)

,title,embedding,distance_to_How High The Moon,similarity_to_How High The Moon,similarity_to_C-Jam Blues,similarity_to_Giant Steps
744,Giant Steps,"[-0.196084, -0.883227, 1.386613, 2.869085, -1....",0.727189,0.823540,0.548088,1.000000
460,Countdown,"[0.446056, -0.345534, 1.070461, 3.09797, -1.55...",0.727189,0.762068,0.528432,0.971927
789,Half Nelson,"[0.223498, -1.128945, 1.615362, 2.893291, -0.8...",0.727189,0.844568,0.585906,0.950992
1792,Sippin' At Bells,"[-1.094481, -0.928625, 1.55838, 2.759888, -1.1...",0.727189,0.805994,0.559845,0.944424
1665,Recordame,"[1.1195, -0.558806, 1.205937, 2.754308, -1.241...",0.727189,0.717214,0.512289,0.931180
1594,Peace,"[-0.4167, -2.257149, 1.691818, 3.649913, -0.78...",0.727189,0.717700,0.389078,0.919790
718,Freight Train,"[-1.571545, -1.092578, 1.806667, 3.054951, -0....",0.727189,0.805324,0.456259,0.916768
278,Blues - Bird,"[-1.669001, -1.014947, 1.82029, 3.159443, -0.8...",0.727189,0.847625,0.562486,0.916426
1726,Satellite,"[-0.415876, -0.408823, 2.003074, 2.947204, -1....",0.727189,0.885054,0.574702,0.913929
408,Children Of The Night,"[0.985339, -0.470104, 0.944487, 1.700279, -1.5...",0.727189,0.745177,0.589861,0.913786
